#### Subjects
 - 50000R
 - 50017L
 - 50034R
#### Parameters
 - 1.5 layars of T10's
 - d0 = 5
#### Poses
 - Flexion
 - Extension
 - abduction
 - adduction
 - pinch load
#### End criteria
 - 0.8 mm
 - 50 N

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from phd_helpers.AbaqusPostprocessing import (
    inp2pv, get_field_path, get_field_df, add_field_to_mesh, parse_wallclock_time, parse_final_step_time, get_history_path
)
from phd_helpers.AbaqusPreprocessing import parse_memory_estimate

In [7]:
DIR = Path('../../../../Computational/InpPipeline/outputs/initialFEAstuff/poses_d5CAsubs')

bones = ['tpm']
subjects = ['50000R']#, '50017L', '50034R']
ids = [
    ('d5', '0'), # (run_id_mesh, run_id)
    ]
poses = ['flexion', 'extension', 'abduction', 'adduction', 'pinch_load']
step = 0
frame = -1

field_metrics = ["CPRESS", "U"]


data = []
for subject in subjects:
    for bone in bones:
        instance = f"{bone.upper()}_INST"
        for pose in poses:
            for run_id_mesh, run_id in ids:
                job_name = Path(f'{run_id_mesh}-{pose}-{run_id}')

                path_inp = DIR / f'inpFiles/{subject}/inp'
                path_job = path_inp / job_name
                path_csv = path_job / 'resultCSVs'
                input_path = path_job / job_name.with_suffix('.inp')
                dat_path = path_job / job_name.with_suffix('.dat')
                sta_path = path_job / job_name.with_suffix('.sta')

                # RESULTS #
                meshes = inp2pv(input_path)
                mesh = meshes[bone]

                # Field data
                for metric in field_metrics:
                    field_path = get_field_path(path_csv, metric, step, frame, instance)
                    field_df = get_field_df(field_path)
                    add_field_to_mesh(mesh, field_df)
                mesh['Umag'] = np.linalg.norm(np.column_stack((mesh['U1'], mesh['U2'], mesh['U3'])), axis=1)

                # History data
                history_data = pd.read_csv(get_history_path(path_csv, step))
                RF_data = history_data[history_data['historyOutputKey']=='RF1']
                CAREA_data = history_data[history_data['historyOutputDescription']=='Total area in contact']


                data.append({
                    'subject': subject,
                    'job': job_name,
                    'P': mesh['CPRESS'].max(),
                    'A': CAREA_data['value'].iloc[frame],
                    'F' : np.abs(RF_data['value'].iloc[frame]),
                    'd' : parse_final_step_time(sta_path),
                    'nodes' : mesh.n_points,
                    'elements' : mesh.n_cells,
                    'memory' : parse_memory_estimate(dat_path)['memory_to_minimize_io_gb'],
                    'runtime' : parse_wallclock_time(dat_path),
                })

df = pd.DataFrame(data)

In [8]:
df

,subject,job,P,A,F,d,nodes,elements,memory,runtime
0,50000R,d5-flexion-0,2.825066,56.927898,49.970608,0.140,84193,57027,9.727,2153.0
1,50000R,d5-extension-0,1.978922,42.436710,34.708260,0.203,84193,57027,9.725,5544.0
2,50000R,d5-abduction-0,3.535621,50.648445,49.999218,0.192,84193,57027,9.726,4110.0
3,50000R,d5-adduction-0,3.134415,52.926769,47.974667,0.597,84193,57027,9.723,8413.0
4,50000R,d5-pinch_load-0,3.832304,37.547939,34.208389,0.283,84193,57027,9.725,7162.0


## Plan:
tweak params for extension (50000R)
 - Loop over:
    - max step
    - extrapolation
    - convert_sdi
    - friction
    - hybrid modified

    - stabilisation ?
    - Contact normal behavior ?

In [14]:
#mesh_path = '../../../../Computational/MeshPipeline/outputs/ParamOptimisation/optimise_d0/d5best'
import subprocess
inpMain_path = '../../../../Computational/InpPipeline/main.py'
subprocess.run(['python', inpMain_path])


Updating parameters.json
	Wrote /Users/maro/Library/CloudStorage/OneDrive-UniversityofLeeds/PhD-wrist/WorkPackages/WorkPackages/TMCJ-Contact/Computational/inpPipeline/set_parameters/parameters.json
Full parameter file saved to outputs/tweak_50000R_ext/study1-inp/params/full_params.json

SUBJECT: 50000R
	MESH: d5
		RUN ID: 0
			Runtime: 10.529s - ok
		RUN ID: 1
			Runtime: 10.531s - ok
		RUN ID: 2
			Runtime: 10.609s - ok
		RUN ID: 3
			Runtime: 10.252s - ok
		RUN ID: 4
			Runtime: 10.263s - ok
		RUN ID: 5
			Runtime: 10.345s - ok
		RUN ID: 6
			Runtime: 10.352s - ok
		RUN ID: 7
			Runtime: 10.477s - ok
		RUN ID: 8
			Runtime: 10.497s - ok
		RUN ID: 9
			Runtime: 10.474s - ok
		RUN ID: 10
			Runtime: 10.524s - ok
		RUN ID: 11
			Runtime: 10.549s - ok
		RUN ID: 12
			Runtime: 10.326s - ok
		RUN ID: 13
			Runtime: 10.320s - ok
		RUN ID: 14
			Runtime: 10.286s - ok
		RUN ID: 15
			Runtime: 10.473s - ok
		RUN ID: 16
			Runtime: 10.302s - ok
		RUN ID: 17
			Runtime: 10.327s - ok
		RUN ID: 1

CompletedProcess(args=['python', '../../../../Computational/InpPipeline/main.py'], returncode=0)